# 09 — Silver integrado: merge + epi

Cruza los casos confirmados con meteo+socio: CSV histórico hasta 2024 y Sala Situacional desde 2025.
Luego el actualizador llena solo los componentes faltantes y escribe `data/silver/integrado/meteo_socio_epi_piura_2017_2025.csv`.
Este dataset todavía no tiene feature engineering, por eso no va en gold.

In [ ]:
import sys
from pathlib import Path

In [ ]:
import pandas as pd
from denguekay.config.paths import SILVER_EPI_HISTORICAL, SILVER_SALA, SILVER_METEO_SOCIO, SILVER_INTEGRATED_BASE, SILVER_INTEGRATED
from denguekay.data.processing.merge import merge_with_epi

df_meteo_socio = pd.read_csv(SILVER_METEO_SOCIO, dtype={"ubigeo": "string"})
df_casos = pd.read_csv(SILVER_EPI_HISTORICAL, dtype={"ubigeo": "string"})


In [ ]:
df_sala = pd.read_csv(SILVER_SALA, dtype={"ubigeo": "string"})
df_integrado = merge_with_epi(df_meteo_socio, df_casos, sala_cases=df_sala)
df_integrado.info()

In [ ]:
import subprocess
from denguekay.config.paths import ROOT

SILVER_INTEGRATED.parent.mkdir(parents=True, exist_ok=True)
base_path = SILVER_INTEGRATED_BASE
output_path = SILVER_INTEGRATED
df_integrado.to_csv(base_path, index=False)
subprocess.run(
    [sys.executable, "-m", "denguekay.pipelines.weekly_update", "--input", str(base_path), "--output", str(output_path)],
    cwd=ROOT, check=True,
)
print(f"Guardado en {output_path}")


In [ ]:
df = pd.read_csv(SILVER_INTEGRATED, dtype={"ubigeo": "string"})
print(df['ubigeo'].dtype)          # should be string
print(df['ubigeo'].str.len().unique())  # should be [6]
print(df['casos_Dengue'].dtype)    # should be Int64